# Reading Between the Lines
## Learning from Residuals between Observed and Synthetic Stellar Spectra

Author: Sven Buder (ANU), David Hogg (NYU)  
GitHub repository: https://github.com/svenbuder/residual_cannon  
arXiv: YYMM.NNNNN

In [ ]:
# Setting a new & isolated conda environment for this project in terminal (note specific numpy version):
# conda create -n py_cannon python=3.12
# conda activate py_cannon
# pip install matplotlib astropy
# pip install "numpy==2.2.*"
# pip install git+https://github.com/ANU-RSAA/AnniesLasso.git

In [ ]:
# Importing packages and software version reporting
try:
    %matplotlib inline
    %config InlineBackend.figure_format='retina'
except:
    pass

import sys, warnings; print("python version:", sys.version)

import numpy as np; print('numpy version: ',np.__version__)

import scipy; print('scipy version: ',scipy.__version__)

import IPython; print('IPython version: ',IPython.__version__)

import matplotlib; print('matplotlib version: ',matplotlib.__version__)
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator

import astropy; print('astropy version: ',astropy.__version__)
from astropy.table import Table, join
import astropy.units as u
from astropy.units import UnitsWarning

import thecannon as tc; print('thecannon version: ',tc.__version__)

# Reporting software versions to a .tex file for inclusion in the paper
np.savetxt('tex_text/software_info.tex',
              [f"\\textsc{{python}} (version {sys.version.split()[0]}) and included its packages",
                f"\\textsc{{astropy}} \\citep[v. {astropy.__version__};]{{Robitaille2013,PriceWhelan2018}},",
                f"\\textsc{{IPython}} \\citep[v. {IPython.__version__};]{{ipython}},",
                f"\\textsc{{matplotlib}} \\citep[v. {matplotlib.__version__};]{{matplotlib}},",
                f"\\textsc{{NumPy}} \\citep[v. {np.__version__};]{{numpy}},",
                f"\\textsc{{scipy}} \\citep[v. {scipy.__version__};]{{Scipy}};",
                f"\\textsc{{topcat}} \\citep[v. 4.7;]{{Taylor2005}};",
                f"\\textsc{{thecannon}} \\citep[v. {tc.__version__} from \\url{{https://github.com/ANU-RSAA/AnniesLasso}} maintained by Marc White from the ANU RSAA/AITC's Software Group]{{Ness2015, Casey2016}}."],
              fmt="%s"
)
print('Saved software versions to tex_text/software_info.tex')

In [ ]:
# rerun_everything = True
rerun_everything = False

# 2 Data

## 2.1 GALAH DR4 spectra and labels

In [ ]:
# Read in example spectrum and calculate sigma-clipped residuals as well as chi2 mask

if rerun_everything:
    example_spectrum = Table.read('spectra/170710/170710003201361/170710003201361_allstar_fit_spectrum.fits')

    # minimum cut
    bad_flux = example_spectrum['sob'] > 1.05
    example_spectrum['sob'][bad_flux] = example_spectrum['smod'][bad_flux]

    # significance
    sigma_1_pos = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] > 1
    sigma_1_neg = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] < -1
    sigma_3_pos = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] > 3
    sigma_3_neg = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] < -3

    # clipped residuals
    residuals_1 = example_spectrum['sob'] - example_spectrum['smod']
    residuals_1[sigma_1_pos] = example_spectrum['uob'][sigma_1_pos]
    residuals_1[sigma_1_neg] = -example_spectrum['uob'][sigma_1_neg]
    residuals_3 = example_spectrum['sob'] - example_spectrum['smod']
    residuals_3[sigma_3_pos] = 3*example_spectrum['uob'][sigma_3_pos]
    residuals_3[sigma_3_neg] = -3*example_spectrum['uob'][sigma_3_neg]

    mask_start_end = []

    found_start = False
    found_end = False

    for index in range(len(example_spectrum['wave'])-1):
        if found_start == False:
            if example_spectrum['mob'][index] == False:
                start = index
                found_start = True
        if found_start == True and found_end == False:
            if example_spectrum['mob'][index] == True:
                end = index
                found_end = True
        if found_start == True and found_end == True:
            mask_start_end.append((example_spectrum['wave'][start], example_spectrum['wave'][end]))
            found_start = False
            found_end = False

In [ ]:
# Example Spectrum Plot

if rerun_everything:
    f, gs = plt.subplots(4,1,figsize=(5,7),sharey=True)

    mask_used = False
    for ccd in [1, 2, 3, 4]:

        ax = gs[ccd-1]
        in_ccd = ((example_spectrum["wave"] > (3 + ccd) * 1000) & (example_spectrum["wave"] < (4 + ccd) * 1000))

        ax.plot(example_spectrum['wave'][in_ccd], example_spectrum['sob'][in_ccd], 'k', label='Data', lw = 0.75)
        ax.plot(example_spectrum['wave'][in_ccd], example_spectrum['smod'][in_ccd], 'C0', label='Model', lw = 0.5)

        colors = ['navajowhite', 'orange', 'darkred']

        ax.fill_between(example_spectrum['wave'][in_ccd], 
            np.zeros(np.shape(example_spectrum['sob'][in_ccd])[0]), 
            (example_spectrum['sob'][in_ccd] - example_spectrum['smod'][in_ccd]),
            color = colors[2],label=r'$\Delta f > 3\sigma$',lw=0.75
        )
        ax.fill_between(example_spectrum['wave'][in_ccd], 
            np.zeros(np.shape(example_spectrum['sob'][in_ccd])[0]), 
            residuals_3[in_ccd],
            color = colors[1],label=r'$\Delta f > 2\sigma$',lw=0.75
        )
        ax.fill_between(example_spectrum['wave'][in_ccd], 
            np.zeros(np.shape(example_spectrum['sob'][in_ccd])[0]), 
            residuals_1[in_ccd],
            color = colors[0],label=r'$\Delta f \leq 1\sigma$',lw=0.75
        )
        
        for mask in mask_start_end:
            if (mask[0] > (3 + ccd) * 1000) & (mask[1] < (4 + ccd) * 1000):
                
                if mask_used:
                    legend = '_nolegend_'
                else:
                    mask_used = True
                    legend = 'DR4 Masked'
                ax.axvspan(mask[0], mask[1], ymin = 0.0, ymax=0.1, color='lightgrey', lw = 0.75, label = legend)#, color='lightgrey', alpha=0.5, label='masked region')
        
        if ccd == 1:
            ax.legend(ncol=3,fontsize=12, loc='upper right', bbox_to_anchor=(1.02, 1.55), frameon=False)
        if ccd == 4:
            ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        ax.set_ylabel(r'Flux $f_\lambda~/~\mathrm{norm.}$', fontsize=12)

    plt.tight_layout(w_pad=0.0, h_pad=0.0)
    plt.savefig('figures/example_spectrum_170710003201361.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Define common wavelength grid for Cannon training and testing
common_wavelength = np.concatenate((
    np.arange(4719.0299,4893.8808,0.0460),
    np.arange(5655.5298,5863.0182,0.0547),
    np.arange(6486.0104,6726.4572,0.0631),
    np.arange(7689.1261,7874.3127,0.0735)
))

## 2.2 Representative Samples

In [ ]:
# Read in GALAH DR4 and down-size to 170710 - 170712

if rerun_everything:
    dr4 = Table.read('data/galah_dr4_allstar_240705.fits')
    dr4_tmass_ids = (dr4[['tmass_id']]).copy()
    dr4['date'] = np.array([str(dr4['sobject_id'][x])[:6] for x in range(len(dr4['sobject_id']))])
    dr4 = dr4[(dr4['date'] >= '170710') & (dr4['date'] <= '170712')]
    print(len(dr4), 'stars in 170710 - 170712')

### 2.2.1 Baseline Sample

In [ ]:
# Load or select&save baseline sample for Cannon training and testing

if rerun_everything:
    baseline_sample = (
        (dr4['flag_sp'] < 2**14) &
        (dr4['chi2_sp'] < 2) &
        (dr4['snr_px_ccd1'] > 10) & (dr4['snr_px_ccd2'] > 10) & (dr4['snr_px_ccd3'] > 10) & (dr4['snr_px_ccd4'] > 10) &
        np.isfinite(dr4['ebv']) & np.isfinite(dr4['a_ks'])
    )
    print(len(dr4[baseline_sample]), 'stars in baseline sample')

    # fast rotators and hot stars (to show as excluded in abundance sample)
    fast_rotators_and_hot_stars = dr4[(
        baseline_sample &
        (
            (dr4['vsini'] > 15) | (dr4['teff'] > 6500)
        )
    )]

    baseline_residual_flux = []
    baseline_residual_ivar = []

    # Interpolate onto same wavelength
    for sobject_id in dr4['sobject_id'][baseline_sample]:
        
        spectrum = Table.read('spectra/'+str(sobject_id)[:6]+'/'+str(sobject_id)+'/'+str(sobject_id)+'_allstar_fit_spectrum.fits')

        baseline_residual_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']-spectrum['smod']))
        baseline_residual_ivar.append(1./(np.interp(common_wavelength,spectrum['wave'],spectrum['uob']))**2)

    baseline_residual_flux = np.array(baseline_residual_flux)
    baseline_residual_ivar = np.array(baseline_residual_ivar)

    np.savez('data/flux_ivar_labels_model1.npz', flux=baseline_residual_flux, ivar=baseline_residual_ivar, labels=dr4[['sobject_id','teff','logg','fe_h']][baseline_sample])
    np.savez('data/flux_ivar_labels_model3.npz', flux=baseline_residual_flux, ivar=baseline_residual_ivar, labels=dr4[['sobject_id','teff','logg','fe_h','a_ks']][baseline_sample])

model1_data = np.load('data/flux_ivar_labels_model1.npz')
print('Loaded flux, ivar, and labels for model 1 with '+str(np.shape(model1_data['flux']))+' spectra and labels '+str(list(model1_data['labels'].dtype.names)))

model3_data = np.load('data/flux_ivar_labels_model3.npz')
print('Loaded flux, ivar, and labels for model 3 with '+str(np.shape(model3_data['flux']))+' spectra and labels '+str(list(model3_data['labels'].dtype.names)))

In [ ]:
# Show Teff-logg diagram of the baseline sample

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))
    s = ax.scatter(
        model1_data['labels']['teff'],
        model1_data['labels']['logg'],
        c = model1_data['labels']['fe_h'],
        cmap = 'RdYlBu',
        s = 4,
        vmin = -1.5, vmax = 0.5
    )
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$',fontsize=15)
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}})$',fontsize=15)
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.text(0.05,0.95,'Models 1 & 3: ' + str(len(model1_data['labels']['sobject_id'])) + ' Stars',va='top',ha='left',transform=ax.transAxes,fontsize=15)
    cbar = plt.colorbar(s,ax=ax)
    cbar.set_label('[Fe/H]',fontsize=15)

    ax.set_xlim(8250, 3500)
    ax.set_ylim(5.1,0.4)

    plt.tight_layout()
    plt.savefig('figures/labels_m1_teff_logg_feh.png',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

### 2.2.2 Abundance Sample

In [ ]:
# Load or select&save abundance sample for Cannon training and testing

if rerun_everything:
    abundance_sample = (
        baseline_sample &
        (dr4['snr_px_ccd1'] > 25) & (dr4['snr_px_ccd2'] > 25) & (dr4['snr_px_ccd3'] > 25) & (dr4['snr_px_ccd4'] > 25) &
        (dr4['vsini'] < 15) &
        (dr4['teff'] < 6500) &
        (dr4['flag_mg_fe'] == 0) & # alpha-proces
        (dr4['flag_na_fe'] == 0) & # odd-Z; more measurements than Al
        (dr4['flag_mn_fe'] == 0) & # iron-peak
        (dr4['flag_ba_fe'] == 0) # s-process; more measurements than Y
    )
    print(len(dr4[abundance_sample]), 'stars in abundance sample')

    abundance_residual_flux = []
    abundance_residual_ivar = []

    # Interpolate onto same wavelength
    for sobject_id in dr4['sobject_id'][abundance_sample]:
        
        spectrum = Table.read('spectra/'+str(sobject_id)[:6]+'/'+str(sobject_id)+'/'+str(sobject_id)+'_allstar_fit_spectrum.fits')

        abundance_residual_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']-spectrum['smod']))
        abundance_residual_ivar.append(1./(np.interp(common_wavelength,spectrum['wave'],spectrum['uob']))**2)

    abundance_residual_flux = np.array(abundance_residual_flux)
    abundance_residual_ivar = np.array(abundance_residual_ivar)

    np.savez('data/flux_ivar_labels_model2.npz', flux=abundance_residual_flux, ivar=abundance_residual_ivar, labels=dr4[['sobject_id','teff','logg','fe_h','na_fe','mg_fe','mn_fe','ba_fe']][abundance_sample])

model2_data = np.load('data/flux_ivar_labels_model2.npz')
print('Loaded flux, ivar, and labels for model 2 with '+str(np.shape(model2_data['flux']))+' spectra and labels '+str(list(model2_data['labels'].dtype.names)))

In [ ]:
# Show Teff-logg diagram of the abundance sample

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))

    # plot neglected fast rotators and hot stars
    s = ax.scatter(
        fast_rotators_and_hot_stars['teff'],
        fast_rotators_and_hot_stars['logg'],
        marker='x',
        s=3,
        linewidths=0.5,
        alpha = 0.5,
        color = 'grey',
        zorder = 1
    )
    s = ax.scatter(
        model2_data['labels']['teff'],
        model2_data['labels']['logg'],
        c = model2_data['labels']['fe_h'],
        cmap = 'RdYlBu',
        s = 4,
        vmin = -1.5, vmax = 0.5,
        zorder = 2
    )
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$',fontsize=15)
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}})$',fontsize=15)
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.text(0.05,0.95,'Model 2: ' + str(len(model2_data['labels']['sobject_id'])) + ' Stars',va='top',ha='left',transform=ax.transAxes,fontsize=15)
    cbar = plt.colorbar(s,ax=ax)
    cbar.set_label('[Fe/H]',fontsize=15)

    ax.text(8000,4.4,'Excluded\nHot Stars and Fast Rotators',ha='left',va='top',fontsize=8,color='grey')

    ax.set_xlim(8250, 3500)
    ax.set_ylim(5.1,0.4)

    # inset with cumulative distribution oftypical absolute residuals
    axins = ax.inset_axes([0.175, 0.5, 0.35, 0.35])
    axins.set_xlabel('[Fe/H]',fontsize=10)
    axins.set_ylabel('[Mg/Fe]',fontsize=10)

    feh_lim = (-2.35,0.65)
    xfe_lim = (-0.2,0.65)
    axins.set_xlim(feh_lim)
    axins.set_ylim(xfe_lim)

    axins.scatter(
        model2_data['labels']['fe_h'],
        model2_data['labels']['mg_fe'],
        c = 'k',
        s = 2,
        lw = 0.
    )
    axins.hist2d(
        model2_data['labels']['fe_h'],
        model2_data['labels']['mg_fe'],
        bins = (np.linspace(feh_lim[0], feh_lim[1], 50), np.linspace(xfe_lim[0], xfe_lim[1], 50)),
        cmap = 'Greys_r',
        cmin = 5,
        norm = LogNorm()
    )
    axins.tick_params(axis='both', labelsize=9)
    axins.set_xticks([-2, -1, 0])
    axins.set_yticks([0, 0.25, 0.5])

    plt.tight_layout()
    plt.savefig('figures/labels_m2_teff_logg_feh_xfe.png',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

### 2.2.3 Activity Sample

In [ ]:
# Crossmatch with activity studies by Boro Saikia+ 2018, Brown+ 2022, Gomes da Silva+ 2021, Lorenzo Oliveira+ 2018, and Zhang+ 2024

if rerun_everything:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UnitsWarning)

        # We have x-matched the following catalogues with 2MASS via 5" RA/DEC:
        rhk_literature = dict()
        rhk_literature['BoroSaikia2018_AandA_616_108'] = Table.read('data/BoroSaikia2018_AandA_616_108_2MASS.fits')
        rhk_literature['Brown2022_MNRAS_514_4300'] = Table.read('data/Brown2022_MNRAS_514_4300_2MASS.fits')
        rhk_literature['GomesDaSilva_2021_AandA_646_77'] = Table.read('data/GomesDaSilva_2021_AandA_646_77_2MASS.fits')
        rhk_literature['LorenzoOliveira_2018_AandA_619_73'] = Table.read('data/LorenzoOliveira_2018_AandA_619_73_2MASS.fits')
        rhk_literature['Zhang_2024_AandA_688_23'] = Table.read('data/Zhang_2024_AandA_688_23_GALAH_DR4.fits')

        # Now let's crossmatch with GALAH DR4 via 2MASS IDs
        for rhk_source in rhk_literature.keys():
            try:
                rhk_literature[rhk_source].rename_column('2MASS','tmass_id')
            except:
                pass

            dr4_rhk = join(rhk_literature[rhk_source], dr4_tmass_ids, keys='tmass_id',metadata_conflicts='silent')
            print('Overlap with stellar activity study by '+rhk_source+':',len(dr4_rhk))


In [ ]:
# Load or select&save activity sample (baseline sample x Zhang+ 2024) for Cannon training and testing

if rerun_everything:

    Zhang_2024_AandA_688_23 = Table.read('data/Zhang_2024_AandA_688_23_GALAH_DR4.fits')
    Zhang_2024_AandA_688_23.rename_column('logRHKcl', 'logRHK')

    activity_sample = join(dr4[baseline_sample], Zhang_2024_AandA_688_23, keys = 'sobject_id', metadata_conflicts='silent')

    # Get rid of the 1 star with significantly lower logg = 3.4
    activity_sample = activity_sample[activity_sample['logg'] > 3.8]

    print(len(activity_sample), 'stars in activity sample')

    activity_residual_flux = []
    activity_residual_ivar = []

    # Interpolate onto same wavelength
    for sobject_id in activity_sample['sobject_id']:
        
        spectrum = Table.read('spectra/'+str(sobject_id)[:6]+'/'+str(sobject_id)+'/'+str(sobject_id)+'_allstar_fit_spectrum.fits')

        activity_residual_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']-spectrum['smod']))
        activity_residual_ivar.append(1./(np.interp(common_wavelength,spectrum['wave'],spectrum['uob']))**2)

    activity_residual_flux = np.array(activity_residual_flux)
    activity_residual_ivar = np.array(activity_residual_ivar)

    np.savez('data/flux_ivar_labels_model4.npz', flux=activity_residual_flux, ivar=activity_residual_ivar, labels=activity_sample[['sobject_id','teff','logg','fe_h','logRHK']])

model4_data = np.load('data/flux_ivar_labels_model4.npz')
print('Loaded flux, ivar, and labels for model 4 with '+str(np.shape(model4_data['flux']))+' spectra and labels '+str(list(model4_data['labels'].dtype.names)))

In [ ]:
# Show Teff-logg diagram of the activity sample

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))
    s = ax.scatter(
        model4_data['labels']['teff'],
        model4_data['labels']['logg'],
        c = model4_data['labels']['logRHK'],
        cmap = 'RdYlBu',
        s = 4
    )
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$',fontsize=15)
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}})$',fontsize=15)
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.text(0.05,0.95,'Model 4: ' + str(len(model4_data['labels']['sobject_id'])) + ' Stars',va='top',ha='left',transform=ax.transAxes,fontsize=15)
    cbar = plt.colorbar(s,ax=ax)
    cbar.set_label(r"$\log R'_\mathrm{HK}$",fontsize=15)

    axins = ax.inset_axes([0.66, 0.66, 0.33, 0.33])
    axins.hist(activity_sample['logRHK'], bins=np.linspace(-5.,-3.9,16), histtype="step")
    axins.set_xlabel(r"$\log R'_\mathrm{HK}$", fontsize=12)
    axins.set_ylabel("Nr", fontsize=12)

    ax.set_xlim(8250, 3500)
    ax.set_ylim(5.1,0.4)

    plt.tight_layout()
    plt.savefig('figures/labels_m4_teff_logg_logrhk.png',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

# 3 Analysis

## 3.1 Residual Analysis

In [ ]:
# Calculate absolute residual flux percentiles & their significance for model 1 & how often residuals are above/below +/- a chosen sigma

residual_flux_percentiles = np.percentile(model1_data['flux'], q=[16,50,84], axis=0)

absolute_residual_flux_percentiles = np.percentile(np.abs(model1_data['flux']), q=[16,50,84], axis=0)

significance_residual_flux_percentiles = np.percentile(np.abs(model1_data['flux']) * np.sqrt(model1_data['ivar']), q=[16,50,84], axis=0)

In [ ]:
# Plot absolute residual flux percentiles & their significance for model 1

if rerun_everything:
    f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)
    for ccd in [1, 2, 3, 4]:
        
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.plot(common_wavelength[in_ccd],residual_flux_percentiles[1][in_ccd], 'k', label='Median Residual', lw = 0.75)
        ax.fill_between(common_wavelength[in_ccd], residual_flux_percentiles[0][in_ccd], residual_flux_percentiles[2][in_ccd], color = 'navajowhite', label=r'16th-84th Percentile', lw=0.75)

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.set_ylabel(r'Residuals $\Delta f_\lambda$', fontsize=12)
            ax.legend(fontsize=12, loc='upper right', frameon=False)

    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig('figures/residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)
    for ccd in [1, 2, 3, 4]:
        
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.plot(common_wavelength[in_ccd],absolute_residual_flux_percentiles[1][in_ccd], 'k', label='Median Residual', lw = 0.75)
        ax.fill_between(common_wavelength[in_ccd], absolute_residual_flux_percentiles[0][in_ccd], absolute_residual_flux_percentiles[2][in_ccd], color = 'navajowhite', label=r'16th-84th Percentile', lw=0.75)

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.set_ylabel(r'Residuals $\vert \Delta f_\lambda \vert$', fontsize=12)
            ax.legend(fontsize=12, loc='upper right', frameon=False)

    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig('figures/absolute_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)
    for ccd in [1, 2, 3, 4]:
        
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.plot(common_wavelength[in_ccd],significance_residual_flux_percentiles[1][in_ccd], 'k', label='Median Residual', lw = 0.75)
        ax.fill_between(common_wavelength[in_ccd], significance_residual_flux_percentiles[0][in_ccd], significance_residual_flux_percentiles[2][in_ccd], color = 'navajowhite', label=r'16th-84th Percentile', lw=0.75)

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.set_ylabel(r'Significance of $\vert \Delta f_\lambda \vert$', fontsize=12)
            ax.legend(fontsize=12, loc='upper right', frameon=False)

        ax.set_yscale('log')

    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig('figures/significance_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Plot how often residuals are above/below +/- a 1/2/3 sigma for model 1

if rerun_everything:
    for significance in [1,2,3]:

        # How often is residual above +X sigma?
        residuals_above_significance = model1_data['flux'] * np.sqrt(model1_data['ivar']) > significance
        percentage_above_significance = 100 * np.mean(residuals_above_significance, axis=0)

        # How often is residual below -X sigma?
        residuals_below_significance = model1_data['flux'] * np.sqrt(model1_data['ivar']) < -significance
        percentage_below_significance = 100 * np.mean(residuals_below_significance, axis=0)

        f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)

        for ccd in [1, 2, 3, 4]:
            
            ax = gs[ccd-1]

            in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

            ax.fill_between(
                common_wavelength[in_ccd],
                np.zeros(len(common_wavelength[in_ccd])),
                percentage_above_significance[in_ccd],
                color = 'lightblue'#, 'C0', label='Too high', lw = 0.75
            )
            ax.fill_between(
                common_wavelength[in_ccd],
                np.zeros(len(common_wavelength[in_ccd])),
                -percentage_below_significance[in_ccd],
                color = 'lightcoral'#, 'C3', label='Too low', lw = 0.75
            )
            line1, = ax.plot(common_wavelength[in_ccd],percentage_above_significance[in_ccd], 'C0', label='Observation $>' + str(significance) + r'\sigma$ shallower', lw = 0.75)
            line2, = ax.plot(common_wavelength[in_ccd],-percentage_below_significance[in_ccd], 'C3', label='Observation $>' + str(significance) + r'\sigma$ deeper', lw = 0.75)

            ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
            ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
            if ccd == 1:
                ax.set_ylabel('Percentage of spectra\n above ' + r'$'+ str(significance) + r'\sigma$ or below ' + r'$-'+ str(significance) + r'\sigma$', fontsize=12)
            if ccd == 1:
                legend1 = ax.legend(handles=[line1], fontsize=12, loc='upper center', borderpad=0.25, edgecolor='none', framealpha=0.5)
                ax.add_artist(legend1)
                legend2 = ax.legend(handles=[line2], fontsize=12, loc='lower center', borderpad=0.25, edgecolor='none', framealpha=0.5)
                ax.add_artist(legend2)

        plt.tight_layout(w_pad=0, h_pad=0)
        plt.savefig(f'figures/percentage_{significance}sigma_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

In [ ]:
# Plot cumulative distribution of significance of residual flux; with an inset of the typical residuals.

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))

    ax.ecdf(significance_residual_flux_percentiles[1].clip(max=4), label='Median Residual', c = 'k')
    ax.ecdf(significance_residual_flux_percentiles[0].clip(max=4), label='16th Percentile', c = 'navajowhite')
    ax.ecdf(significance_residual_flux_percentiles[2].clip(max=4), label='84th Percentile', c = 'lightcoral')
    ax.text(4.0,0.95,r'clipped at 4$\sigma$',va='top',ha='right',fontsize=10)
    ax.set_xlabel(r'Significance of $\vert \Delta f_\lambda \vert$', fontsize=12)
    ax.set_ylabel('Cumulative Distribution', fontsize=12)
    ax.legend(fontsize=12, loc=(0.6,0.554), frameon=False)

    ax.set_xlim(0, 4.05)
    ax.set_ylim(0, 1.0)

    x = 0.878
    ax.axvline(1,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.axhline(x,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.text(1.01, x-0.01, '(1,0.878)', ha='left', va='top', fontsize=10)

    x = 0.9734
    ax.axvline(2,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.axhline(x,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.text(2.01, x-0.01, '(2,0.973)', ha='left', va='top', fontsize=10)

    ax.axvline(3,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.axvline(4,color='grey', ls='--', lw=0.75, alpha=0.5)

    # inset with cumulative distribution oftypical absolute residuals
    axins = ax.inset_axes([0.617, 0.15, 0.37, 0.37])
    axins.ecdf(absolute_residual_flux_percentiles[1].clip(max=0.1), label='Median Residual', c = 'k')
    axins.ecdf(absolute_residual_flux_percentiles[0].clip(max=0.1), label='16th Percentile', c = 'navajowhite')
    axins.ecdf(absolute_residual_flux_percentiles[2].clip(max=0.1), label='84th Percentile', c = 'lightcoral')
    axins.text(0.10,0.95,'clipped\nat 0.1',va='top',ha='right',fontsize=8)
    axins.set_xlabel(r'Absolute Residuals $\vert \Delta f_\lambda \vert$', fontsize=10)
    axins.set_ylabel(r'Cumulative Distribution', fontsize=10)
    axins.legend(fontsize=7, loc='lower right', frameon=False)
    axins.tick_params(axis='both', labelsize=9)    
    axins.set_xlim(0, 0.105)
    axins.set_yticks([0,0.5,1])
    axins.set_ylim(0, 1.0)
    axins.axhline(0.5, color='grey', ls='--', lw=0.75, alpha=0.5)

    plt.tight_layout()
    plt.savefig('figures/cdf_significance_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

## 3.2 Residual Cannon Models

In [ ]:
cannon_models = dict()

tests_run = [1,2,3,4]

# Populate the relevant dictionary of input data
for test in tests_run:

    if test == 1: data_to_use = model1_data
    if test == 2: data_to_use = model2_data
    if test == 3: data_to_use = model3_data
    if test == 4: data_to_use = model4_data

    label_names = list(data_to_use['labels'].dtype.names[1:])

    cannon_models['model_'+str(test)] = dict()
    cannon_models['model_'+str(test)]['label_names'] = label_names
    cannon_models['model_'+str(test)]['sobject_id'] = data_to_use['labels']['sobject_id']
    cannon_models['model_'+str(test)]['labels'] = np.array([data_to_use['labels'][label] for label in cannon_models['model_'+str(test)]['label_names']]).T
    cannon_models['model_'+str(test)]['flux']  = data_to_use['flux']
    cannon_models['model_'+str(test)]['ivar']  = data_to_use['ivar']

    if rerun_everything:

        # Set up the model
        cannon_model = tc.CannonModel(
            training_set_labels = cannon_models['model_'+str(test)]['labels'],
            training_set_flux = cannon_models['model_'+str(test)]['flux'],
            training_set_ivar = cannon_models['model_'+str(test)]['ivar'],
            vectorizer=tc.vectorizer.PolynomialVectorizer(list(cannon_models['model_'+str(test)]['label_names']), 2),dispersion=common_wavelength)

        # Train the model
        cannon_models['model_'+str(test)]['theta'], cannon_models['model_'+str(test)]['s2'], cannon_models['model_'+str(test)]['metadata'] = cannon_model.train()
        cannon_models['model_'+str(test)]['model'] = cannon_model

        # Save the model
        cannon_model.write('models/cannon_model_m'+str(test)+'.model', overwrite=True)

        # Test the model
        cannon_models['model_'+str(test)]['out'], cannon_models['model_'+str(test)]['cov'], dummy = cannon_model.test(cannon_models['model_'+str(test)]['flux'], cannon_models['model_'+str(test)]['ivar'], threads=1)
        cannon_models['model_'+str(test)]['flux_out'] = np.array(cannon_model.__call__(cannon_models['model_'+str(test)]['out']))

        np.savez('models/cannon_model_m'+str(test)+'_recovery.npz', out=cannon_models['model_'+str(test)]['out'], cov=cannon_models['model_'+str(test)]['cov'], flux_out=cannon_models['model_'+str(test)]['flux_out'], sobject_id=cannon_models['model_'+str(test)]['sobject_id'])

    # Read in the trained model and recovery results
    cannon_model = tc.CannonModel.read('models/cannon_model_m'+str(test)+'.model')
    recovery = np.load('models/cannon_model_m'+str(test)+'_recovery.npz')
    cannon_models['model_'+str(test)]['theta'] = cannon_model.theta
    cannon_models['model_'+str(test)]['s2'] = cannon_model.s2
    cannon_models['model_'+str(test)]['model'] = cannon_model
    cannon_models['model_'+str(test)]['out'] = recovery['out']
    cannon_models['model_'+str(test)]['cov'] = recovery['cov']
    cannon_models['model_'+str(test)]['flux_out'] = recovery['flux_out']

In [ ]:
# Visualize the recovery of labels for each model, using different colormaps for the different versions
for test in tests_run:

    if test == 1: cmap = 'Greys_r'
    if test == 2: cmap = 'Blues_r'
    if test == 3: cmap = 'Oranges_r'
    if test == 4: cmap = 'Purples_r'

    for index, label in enumerate(cannon_models['model_'+str(test)]['label_names']):

        f, ax = plt.subplots(figsize=(4,3))
        
        recovery_data = cannon_models['model_'+str(test)]['out'][:,index] - cannon_models['model_'+str(test)]['labels'][:,index]

        if label == 'teff':
            recovery_data = recovery_data.clip(min=-2000,max=2000)
        elif label == 'logg':
            recovery_data = recovery_data.clip(min=-1.9,max=1.9)
        elif label == 'ba_fe':
            recovery_data = recovery_data.clip(min=-1.,max=1.)
        elif label == 'a_ks':
            recovery_data = recovery_data.clip(min=-0.15,max=0.15)

        input_minmax = [np.min(cannon_models['model_'+str(test)]['labels'][:,index]), np.max(cannon_models['model_'+str(test)]['labels'][:,index])]
        input_minmax = [1.1*input_minmax[0]-0.1*input_minmax[1],1.1*input_minmax[1]-0.1*input_minmax[0]]
        delta_absmax = np.max(np.abs(np.min(recovery_data)))
        delta_minmax = [-1.1*delta_absmax, 1.1*delta_absmax]

        ax.hist2d(
            cannon_models['model_'+str(test)]['labels'][:,index],
            recovery_data,
            cmin = 1,
            cmap = cmap,
            bins = (np.linspace(input_minmax[0],input_minmax[1],100),np.linspace(delta_minmax[0],delta_minmax[1],100)),
            norm = LogNorm()
        )

        if label == 'teff':
            fancy_label = r'$T_\mathrm{eff}~/~\mathrm{K}$'
        elif label == 'logg':
            fancy_label = r'$\log (g~/~\mathrm{cm\,s^{-2}})$'
        elif label == 'fe_h':
            fancy_label = r'$\mathrm{[Fe/H]}$'
        elif label == 'ebv':
            fancy_label = r'$E(B-V)~/~\mathrm{mag}$'
        elif label == 'a_ks':
            fancy_label = r'$A_\mathrm{K_S}~/~\mathrm{mag}$'
        elif label == 'na_fe':
            fancy_label = r'$\mathrm{[Na/Fe]}$'
        elif label == 'mg_fe':
            fancy_label = r'$\mathrm{[Mg/Fe]}$'
        elif label == 'mn_fe':
            fancy_label = r'$\mathrm{[Mn/Fe]}$'
        elif label == 'ba_fe':
            fancy_label = r'$\mathrm{[Ba/Fe]}$'
        elif label == 'logRHK':
            fancy_label = r'$\log R^\prime_\mathrm{HK}$'
        else:
            fancy_label = label

        # Remember the input distribution with significant decimal places
        input_distribution = np.percentile(cannon_models['model_'+str(test)]['labels'][:,index], q=[16,50,84])
        if np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 100:
            fmt = "{:.0f}"
            input_distribution = np.array(np.round(input_distribution,-1),dtype=int)
        elif np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 10:
            fmt = "{:.0f}"
            input_distribution = np.array(np.round(input_distribution,0),dtype=int)
        elif np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 1:
            fmt = "{:.1f}"
            input_distribution = np.round(input_distribution,1)
        elif np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 0.1:
            fmt = "{:.2f}"
            input_distribution = np.round(input_distribution,2)
        else:
            fmt = "{:.3f}"
            input_distribution = np.round(input_distribution,3)

        ax.set_xlabel('Input '+fancy_label[:-1]+ r' \in '+fmt.format(input_distribution[1])+'_{-'+fmt.format(input_distribution[1]-input_distribution[0])+'}^{+'+fmt.format(input_distribution[2]-input_distribution[1])+'}$',fontsize=15)

        # Calculate the percentiles of the residuals with significant decimal places
        percentiles = np.percentile(cannon_models['model_'+str(test)]['out'][:,index] - cannon_models['model_'+str(test)]['labels'][:,index], q = [16,50,84])
        if np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 100:
            fmt = "{:.0f}"
            percentiles = np.array(np.round(percentiles,-1),dtype=int)
        elif np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 10:
            fmt = "{:.0f}"
            percentiles = np.array(np.round(percentiles,0),dtype=int)
        elif np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 1:
            fmt = "{:.1f}"
            percentiles = np.round(percentiles,1)
        elif np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 0.1:
            fmt = "{:.2f}"
            percentiles = np.round(percentiles,2)
        else:#if np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 0.01:
            fmt = "{:.3f}"
            percentiles = np.round(percentiles,3)
        
        ax.set_ylabel(r'$\Delta$'+fancy_label,fontsize=15)
        ax.text(0.05,0.05,r'$\Delta = '+fmt.format(percentiles[1])+'_{-'+fmt.format(percentiles[1]-percentiles[0])+'}^{+'+fmt.format(percentiles[2]-percentiles[1])+'}$',va='bottom',ha='left',transform=ax.transAxes,fontsize=12)

        ax.axhline(0.0,c='k',lw=1,ls='dashed')

        ax.set_aspect('equal', adjustable='datalim')
        
        plt.tight_layout()
        plt.savefig('figures/recovery_model'+str(test)+'_'+label+'.png',dpi=200,bbox_inches='tight')
        plt.show()
        plt.close()

In [ ]:
# Plot linear coefficients for each model in one figure, using different colormaps for the different versions
for test in tests_run:

    if test == 1: cmap = 'Greys_r'; color = 'k'
    if test == 2: cmap = 'Blues_r'; color = 'C0'
    if test == 3: cmap = 'Oranges_r'; color = 'C1'
    if test == 4: cmap = 'Purples_r'; color = 'C4'

    rows = 2+len(cannon_models['model_'+str(test)]['label_names'])

    f, gs = plt.subplots(
        rows,
        4,
        figsize=(10,1*rows),
        # sharey=True
    )
    gs = gs.flatten()

    for index in range(rows):

        if index == 0:
            ylabel = 'Median\n'+r'Model $F_\lambda$'
            labelfontsize = 12
        elif index == 1:
            ylabel = 'Model\nScatter'
            labelfontsize = 12
        else:
            label = cannon_models['model_'+str(test)]['label_names'][index-2]
            labelfontsize = 15
            if label == 'teff':
                fancy_label = r'T_\mathrm{eff}'#r'$T_\mathrm{eff}~/~\mathrm{K}$'
            elif label == 'logg':
                fancy_label = r'\log (g)'#~/~\mathrm{cm\,s^{-2}})$'
            elif label == 'fe_h':
                fancy_label = r'\mathrm{[Fe/H]}'
            elif label == 'na_fe':
                fancy_label = r'\mathrm{[Na/Fe]}'
            elif label == 'mg_fe':
                fancy_label = r'\mathrm{[Mg/Fe]}'
            elif label == 'mn_fe':
                fancy_label = r'\mathrm{[Mn/Fe]}'
            elif label == 'ba_fe':
                fancy_label = r'\mathrm{[Ba/Fe]}'
            elif label == 'ebv':
                fancy_label = r'E(B-V)'#~/~\mathrm{mag}$'
            elif label == 'a_ks':
                fancy_label = r'A_\mathrm{K_S}'#~/~\mathrm{mag}$'
            elif label == 'logRHK':
                fancy_label = r'\log R_\mathrm{HK}^\prime'
            else:
                fancy_label = label

            print(fancy_label)
            ylabel = r'$\frac{\mathrm{d}F_\lambda}{\mathrm{d}'+fancy_label+r'}$'
        
        for ccd in [1,2,3,4]:
            ax = gs[index*4+ccd-1]
            in_ccd = (common_wavelength > (int(ccd)+3)*1000) & (common_wavelength < (int(ccd)+4)*1000)

            if index == 0:
                ydata = cannon_models['model_'+str(test)]['theta'][:,0]
            elif index == 1:
                ydata = np.sqrt(cannon_models['model_'+str(test)]['s2'][:])
            else:
                ydata = cannon_models['model_'+str(test)]['theta'][:,index-1]
                
            ax.plot(
                common_wavelength[in_ccd],
                ydata[in_ccd],
                lw = 0.5,
                c = color
            )
            y_minmax = np.min(ydata),np.max(ydata)

            ax.set_ylim(1.1*y_minmax[0]-0.1*y_minmax[1],1.1*y_minmax[1]-0.1*y_minmax[0])
            ax.set_xlim(common_wavelength[in_ccd][0]-5,common_wavelength[in_ccd][-1]+5)

            if ccd == 1:
                ax.set_ylabel(ylabel,fontsize=labelfontsize)
            else:
                ax.set_yticks([])
            if index == rows-1:
                ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$')
            else:
                ax.set_xticks([])

    f.align_ylabels(gs)
    plt.tight_layout(w_pad=0,h_pad=0)
    plt.savefig('figures/cannon_linear_coefficients_model'+str(test)+'.pdf',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Plot linear coefficients for all models in one figure, using different colormaps for the different versions

rows = 2 + 3+4+1+1

f, gs = plt.subplots(rows,4,figsize=(10,10))
gs = gs.flatten()

for test in tests_run[::-1]:

    if test == 1: cmap = 'Greys_r'; color = 'k'
    if test == 2: cmap = 'Blues_r'; color = 'C0'
    if test == 3: cmap = 'Oranges_r'; color = 'C1'
    if test == 4: cmap = 'Purples_r'; color = 'C4'

    for index in range(rows):

        for ccd in [1,2,3,4]:

            ax = gs[index*4+ccd-1]
            in_ccd = (common_wavelength > (int(ccd)+3)*1000) & (common_wavelength < (int(ccd)+4)*1000)

            # Median models
            if index == 0:
                ylabel = 'Median\n'+r'Model $F_\lambda$'
                labelfontsize = 10
                ydata = cannon_models['model_'+str(test)]['theta'][:,0]
                toplot = True

            # Scatter
            elif index == 1:
                ylabel = 'Model\nScatter'
                labelfontsize = 10
                ydata = np.sqrt(cannon_models['model_'+str(test)]['s2'])
                toplot = True

            else:
                if index == 2:
                    needed_label = 'teff'
                    fancy_label = r'T_\mathrm{eff}'#r'$T_\mathrm{eff}~/~\mathrm{K}$'
                elif index == 3:
                    needed_label = 'logg'
                    fancy_label = r'\log (g)'#~/~\mathrm{cm\,s^{-2}})$'
                elif index == 4:
                    needed_label = 'fe_h'
                    fancy_label = r'\mathrm{[Fe/H]}'
                elif index == 5:
                    needed_label = 'na_fe'
                    fancy_label = r'\mathrm{[Na/Fe]}'
                elif index == 6:
                    needed_label = 'mg_fe'
                    fancy_label = r'\mathrm{[Mg/Fe]}'
                elif index == 7:
                    needed_label = 'mn_fe'
                    fancy_label = r'\mathrm{[Mn/Fe]}'
                elif index == 8:
                    needed_label = 'ba_fe'
                    fancy_label = r'\mathrm{[Ba/Fe]}'
                elif index == 9:
                    needed_label = 'a_ks'
                    fancy_label = r'A_\mathrm{K_S}'
                elif index == 10:
                    needed_label = 'logRHK'
                    fancy_label = r'\log R_\mathrm{HK}^\prime'
                else:
                    raise ValueError('Index '+str(index)+' does not correspond to a known label.')

                if needed_label in cannon_models['model_'+str(test)]['label_names']:
                    label_index = cannon_models['model_'+str(test)]['label_names'].index(needed_label)
                    ydata = cannon_models['model_'+str(test)]['theta'][:,label_index+1]
                    ylabel = r'$\frac{\mathrm{d}F_\lambda}{\mathrm{d}'+fancy_label+r'}$'
                    labelfontsize = 15
                    toplot = True
                else:
                    toplot = False

            if toplot:
                ax.plot(
                    common_wavelength[in_ccd],
                    ydata[in_ccd],
                    lw = 0.3,
                    c = color
                )
                if ccd == 1:
                    ax.set_ylabel(ylabel,fontsize=labelfontsize)
                # else:
                #     ax.set_yticks([])
                if index == rows-1:
                    ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$')
                # else:
                #     ax.set_xticks([])
                y_minmax = np.min(ydata),np.max(ydata)
                ax.set_ylim(1.1*y_minmax[0]-0.1*y_minmax[1],1.1*y_minmax[1]-0.1*y_minmax[0])
                ax.set_xlim(common_wavelength[in_ccd][0]-5,common_wavelength[in_ccd][-1]+5)

for i, ax in enumerate(gs):
    if i % 4 != 0:   # not in first column
        ax.tick_params(axis='y', labelleft=False)
    if i < len(gs) - 4:   # not in last row
        ax.tick_params(axis='x', labelbottom=False)

f.align_ylabels(gs)
plt.tight_layout(w_pad=0,h_pad=-0.35)
plt.savefig('figures/cannon_linear_coefficients_joint.pdf',dpi=200,bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# https://ui.adsabs.harvard.edu/abs/2018AJ....156..180W
Wise2018_lines = [
4827.46,
4861.33,
5707.00,
6562.81
]

# https://ui.adsabs.harvard.edu/abs/2019MNRAS.485.4804L
Lisogorskyi2019_lines = Table.read('data/Lisogorskyi_2019_MNRAS_485_4804_TableA1.fits')
Lisogorskyi2019_lines = Lisogorskyi2019_lines[(
    ((Lisogorskyi2019_lines['center'] >= 4719.0299) & (Lisogorskyi2019_lines['center'] <= 4893.8759)) |
    ((Lisogorskyi2019_lines['center'] >= 5655.5298) & (Lisogorskyi2019_lines['center'] <= 5863.0069)) |
    ((Lisogorskyi2019_lines['center'] >= 6486.0104) & (Lisogorskyi2019_lines['center'] <= 6726.4214)) |
    ((Lisogorskyi2019_lines['center'] >= 7689.1261) & (Lisogorskyi2019_lines['center'] <= 7874.2726))
)]
plt.scatter(
    Lisogorskyi2019_lines['center'],
    Lisogorskyi2019_lines['logS_ew_p'],
)
plt.show()
plt.close()
np.array(Lisogorskyi2019_lines['center'])

In [ ]:
residuals = cannon_models['model_4']['theta'][:,0]

large_residuals = np.where(np.abs(residuals) > 0.075)[0]

count = 0

significant_residuals = []

for residual_index in np.arange(len(large_residuals)-1):
    this_one = common_wavelength[large_residuals][residual_index]
    next_one = common_wavelength[large_residuals][residual_index+1]

    if next_one - this_one > 0.1:
        print(count, this_one, next_one, next_one - this_one)
        significant_residuals.append(this_one)
        count += 1

significant_residuals = np.array(significant_residuals)

In [ ]:
significant_residuals

In [ ]:
residuals = cannon_models['model_4']['theta'][:,5]

large_residuals = np.where(np.abs(residuals) > 0.075)[0]

count = 0

significant_residuals = []

for residual_index in np.arange(len(large_residuals)-1):
    this_one = common_wavelength[large_residuals][residual_index]
    next_one = common_wavelength[large_residuals][residual_index+1]

    if next_one - this_one > 0.1:
        print(count, this_one, next_one, next_one - this_one)
        significant_residuals.append(this_one)
        count += 1

significant_residuals = np.array(significant_residuals)

In [ ]:
investigate = Table()
investigate['lambda'] = common_wavelength
investigate['flux'] = cannon_models['model_4']['theta'][:,0]
investigate['dflux_dteff'] = cannon_models['model_4']['theta'][:,1]
investigate['dflux_dlogg'] = cannon_models['model_4']['theta'][:,2]
investigate['dflux_dfeh'] = cannon_models['model_4']['theta'][:,3]
investigate['dflux_daks'] = cannon_models['model_4']['theta'][:,4]
investigate['dflux_dlogrhk'] = cannon_models['model_4']['theta'][:,5]
investigate.write('data/model_coefficients_test_4.fits',overwrite=True)

In [ ]:
for index, line in enumerate(np.array(Lisogorskyi2019_lines['center'])):

    in_wavelength = (common_wavelength > line - 2) & (common_wavelength < line + 2)

    print(Lisogorskyi2019_lines[index])
    plt.figure(figsize=(10,3))
    plt.plot(
        common_wavelength[in_wavelength],
        cannon_models['model_4']['theta'][in_wavelength,0],
        lw = 0.5
    )
    plt.plot(
        common_wavelength[in_wavelength],
        cannon_models['model_4']['theta'][in_wavelength,5],
        lw = 0.5
    )
    plt.ylim(-0.2,0.2)
    plt.axvline(line)
    plt.show()
    plt.close()

In [ ]:
def plot_fit(star_index, test = 3):

    f, gs = plt.subplots(4,1,figsize=(10,8),sharey=True)
    for ccd in [1,2,3,4]:
        in_ccd = (common_wavelength > (int(ccd)+3)*1000) & (common_wavelength < (int(ccd)+4)*1000)
        ax = gs[ccd-1]
        ax.plot(
            common_wavelength[in_ccd],
            cannon_models['model_'+str(test)]['flux'][star_index,in_ccd],
            c = 'grey', label = 'Residual Flux DR4', lw=0.5
        )
        ax.plot(
            common_wavelength[in_ccd],
            cannon_models['model_'+str(test)]['flux_out'][star_index,in_ccd],
            c = 'C0', label = 'Fit to Residual Flux', lw=0.5
        )
        ax.plot(
            common_wavelength[in_ccd],
            cannon_models['model_'+str(test)]['flux'][star_index,in_ccd] - cannon_models['model_'+str(test)]['flux_out'][star_index,in_ccd] + 0.25,
            c = 'C3', label = 'Residual of Residual + 0.25', lw=0.5
        )
        ax.legend(ncol=3)
    ax.set_ylim(-0.25,0.5)
    f.suptitle(str(star_index)+' '+str(cannon_models['model_'+str(test)]['sobject_id'][star_index])+': TEFF '+str(np.round(cannon_models['model_'+str(test)]['labels'][star_index, 0]))+', log(g) '+str(np.round(cannon_models['model_'+str(test)]['labels'][star_index, 1],2))+', [Fe/H] '+str(np.round(cannon_models['model_'+str(test)]['labels'][star_index, 2],2)))

    plt.tight_layout()
    
    plt.savefig('figures/residuals_test_'+str(test)+'_'+str(cannon_models['model_'+str(test)]['sobject_id'][star_index])+'.pdf',bbox_inches='tight',dpi=200)
    plt.show()
    plt.close()
    
to_plot = [
    0, # first star -> 
    100, # missed lines in CCD1; emission lines in CCD4
    200, # DIBs & KI
]

sobject_ids = [
    # # 170711005801103, # Halpha and Hbeta as well as line at 7800 and 5755
    # # 170711005801334, # missed lines in CCD1; emission lines in CCD4
    # # 170712003101121, # DIBs & KI,
    # # 170710003201320, # highest logRHK
    # 170711003001093,
    # 170711003001103, # high E(B-V) and A(Ks)
    # 170711003001151, # high E(B-V) and A(Ks)
    170711001501201, # cool giant recovered as hot dwarf
]

for sobject_id in sobject_ids:
    for test in tests_run:
        match = np.where(cannon_models['model_'+str(test)]['sobject_id'] == sobject_id)[0]
        if len(match) > 0:
            print(test, sobject_id)
            plot_fit(match[0], test)